# Variational Autoencoders (VAE)

_Modern Deep Learning & AI_

**Encode to a distribution, not a point. Sample it, decode it, and you can generate brand-new data.**

A plain autoencoder maps each input to one point in the code space. Gaps between points decode to garbage, so you cannot generate new samples.

     A variational autoencoder fixes this. The encoder outputs a small distribution for each input: a mean $\mu$ and a spread $\sigma$.

---

This notebook is a practice scaffold for the **Variational Autoencoders (VAE)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, n_in=5, latent=2):
        super().__init__()
        self.enc = nn.Linear(n_in, 8)
        self.mu = nn.Linear(8, latent)
        self.logvar = nn.Linear(8, latent)     # log-variance keeps sigma positive
        self.dec = nn.Sequential(nn.Linear(latent, 8), nn.ReLU(), nn.Linear(8, n_in))
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)          # sigma
        eps = torch.randn_like(std)            # noise ~ N(0, I)
        return mu + std * eps                  # z = mu + sigma * eps
    def forward(self, x):
        h = F.relu(self.enc(x))
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparam(mu, logvar)
        return self.dec(z), mu, logvar

torch.manual_seed(0)
model = VAE()
x = torch.rand(8, 5)                            # synthetic batch
x_hat, mu, logvar = model(x)
recon = F.mse_loss(x_hat, x, reduction="sum")
kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())  # KL to N(0, I)
loss = recon + kl
print("recon:", round(recon.item(), 2), "kl:", round(kl.item(), 2))

## Visualize the data & results

_In the learned latent of real digits, how do classes cluster, and which digits are hardest to reconstruct?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# latent space + per-class reconstruction error on REAL handwritten digits
digits = load_digits()
X = digits.data / 16.0
y = digits.target
pca = PCA(n_components=2)
Z = pca.fit_transform(X)
recon = pca.inverse_transform(Z)
mse = np.mean((X - recon) ** 2, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for c, col in zip([0, 1, 2, 3], ["#c89bff", "#4ea1ff", "#7ee787", "#ffb454"]):
    idx = np.where(y == c)[0][:15]
    axes[0].scatter(Z[idx, 0], Z[idx, 1], color=col, label="digit %d" % c)
axes[0].set_xlabel("latent z1"); axes[0].set_ylabel("latent z2")
axes[0].set_title("real digits 0-3 in latent space"); axes[0].legend()

class_mse = [float(mse[y == c].mean()) for c in range(10)]
axes[1].bar([str(c) for c in range(10)], class_mse, color="#c89bff")
axes[1].set_title("mean MSE per digit class")
plt.show()